# P1 — Data overview: 2024 Pediatric Sepsis Challenge

**Question we're working towards:** *From information available at admission, which children with suspected sepsis will die in hospital?*

**Data:** synthetic training set (2,686 children aged 6–60 months, 6 Ugandan hospitals, 138 variables), generated from the real cohort with CART models (`synthpop`). No row is a real child.
Source: Borealis doi:10.5683/SP3/TFAV36 — licence CC BY-NC-SA 4.0.

**How to use this notebook:** run one cell at a time with **Shift+Enter**. Read the text above each cell *before* running it, then look at the output and compare with the "What to look for" notes.

## 1. Load libraries and data
- `pandas` holds tables (DataFrames); `matplotlib` draws charts.
- The file is **tab-separated**, hence `sep="\t"`.
- `died` is a True/False column we will reuse to compare survivors vs deaths.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("PSDC_SyntheticTrainingData_Dataset_ODR.tab", sep="\t")
died = df["inhospital_mortality"] == 1

print(df.shape)
df.head()

## 2. Structure: unit of analysis and variable groups
**Always check first:** one row = one child? (no duplicate IDs). Then: what kinds of variables do we have?

**What to look for:** 0 duplicates; most columns are text categories (`str`) — they will need encoding before modelling.

In [ ]:
print("Duplicate study IDs:", df["studyid_adm"].duplicated().sum())

dd = pd.read_csv("data_dictionary.txt", sep="|", skipinitialspace=True)
dd.columns = [c.strip() for c in dd.columns]
dd["group"] = dd["Variable Type"].str.extract(r"\((.*)\)")[0].str.strip()
print(dd["group"].value_counts())
print(df.dtypes.astype(str).value_counts())

## 3. The outcome — and why accuracy is useless here
**What to look for:** ~4.4% mortality. A model predicting "everyone survives" would be ~95.6% accurate while missing every death.

⚠️ `lengthadm` (length of stay) is only known at discharge — it is an **outcome**, never a predictor.

In [ ]:
print(df["inhospital_mortality"].value_counts())
print(f"Mortality: {died.mean():.1%}")
print("Accuracy of 'everyone survives':", f"{1 - died.mean():.1%}")

## 4. One child, as the model sees it
Look at a real row before trusting summary numbers.

In [ ]:
df.iloc[0, :25]

## 5. How categories are coded
Three kinds of categorical variables, each encoded differently later:

| Type | Example | Encoding |
|---|---|---|
| Binary | capillary refill ≥3 s | 0/1 |
| Ordinal | Blantyre motor response | ordered numbers |
| Nominal | water source | one-hot columns |

Columns named `name___N` are "check all that apply" tick boxes.

In [ ]:
for col in ["bcsmotor_adm", "bcseye_adm", "feedingstatus_adm", "momedu_adm", "admitabx_adm___8"]:
    print(col, df[col].value_counts(dropna=False).to_dict(), "\n")
print(sum("___" in c for c in df.columns), "tick-box columns")

## 6. Missing data
**What to look for:** low overall; lactate & haematocrit ~17% missing (test not done — informative!); `lactate2` is a *later* lactate → after admission → **leakage**, must be excluded.

In [ ]:
print(f"Overall missing cells: {df.isna().mean().mean():.1%}")
(df.isna().mean() * 100).sort_values(ascending=False).head(8).round(1)

## 7. Plausibility check
Define physiological limits for children 6–60 months and count values outside them.

**What to look for:** glucose = 0 and haematocrit 82% are errors → will be set to missing in P2. SpO₂ 57% is plausible in severe pneumonia → keep.

In [ ]:
limits = {"agecalc_adm": (6, 60), "weight_kg_adm": (3, 30), "muac_mm_adm": (70, 220),
          "hr_bpm_adm": (40, 250), "rr_brpm_app_adm": (8, 120), "sysbp_mmhg_adm": (40, 180),
          "temp_c_adm": (32, 43), "spo2site1_pc_oxi_adm": (40, 100),
          "hematocrit_gpdl_adm": (5, 65), "lactate_mmolpl_adm": (0.3, 25), "glucose_mmolpl_adm": (0.5, 40)}
rows = []
for col, (lo, hi) in limits.items():
    s = df[col]
    rows.append([col, s.min(), s.median(), s.max(), ((s < lo) | (s > hi)).sum(), f"{s.isna().mean():.0%}"])
pd.DataFrame(rows, columns=["variable", "min", "median", "max", "implausible", "missing"]).round(1)

## 8. Clinical signs vs mortality
**What to look for:** Blantyre motor "no response" → ~44% mortality vs ~3% if localises. Oxygen "being used" → higher mortality — a **marker of clinician concern**, not harm (confounding by indication). Malaria-positive → *lower* mortality (treatable cause).

In [ ]:
for col in ["bcsmotor_adm", "respdistress_adm", "caprefill_adm", "hivstatus_adm",
            "malariastatuspos_adm", "oxygenavail_adm"]:
    c = df[col].fillna("missing")
    display(pd.DataFrame({"n": c.value_counts(), "mortality_%": (died.groupby(c).mean() * 100).round(1)}))

## 9. Mortality by age band
**What to look for:** not a straight line (peaks at 12–36 months) → hint that non-linear models or age bands may help.

In [ ]:
band = pd.cut(df["agecalc_adm"], [6, 12, 24, 36, 60], include_lowest=True,
              labels=["6-12 m", "12-24 m", "24-36 m", "36-60 m"])
pd.DataFrame({"children": band.value_counts(sort=False),
              "deaths": died.groupby(band, observed=True).sum(),
              "mortality_%": (died.groupby(band, observed=True).mean() * 100).round(1)})

## 10. Charts
**What to look for:**
- **Weight vs age:** horizontal stripes at 10, 12, 15 kg = *digit preference* (rounding). Deaths (red) sit at the **bottom** of the cloud → low weight **for age** → we need z-scores (P2).
- **Lactate:** deaths shifted right with a long tail >5 mmol/L.
- **Respiratory rate:** heavy overlap → weak on its own.
- Jagged orange histograms = only 119 deaths (small numbers, not biology).

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(15, 8))
for a, (col, title) in zip(ax.flat[:4], [("agecalc_adm", "Age (months)"), ("weight_kg_adm", "Weight (kg)"),
                                         ("rr_brpm_app_adm", "Respiratory rate"), ("lactate_mmolpl_adm", "Lactate (mmol/L)")]):
    a.hist(df.loc[~died, col].dropna(), bins=40, density=True, alpha=.55, label="survived")
    a.hist(df.loc[died, col].dropna(), bins=40, density=True, alpha=.55, label="died")
    a.set_title(title); a.legend()

a = ax.flat[4]
a.scatter(df.loc[~died, "agecalc_adm"], df.loc[~died, "weight_kg_adm"], s=4, alpha=.3, label="survived")
a.scatter(df.loc[died, "agecalc_adm"], df.loc[died, "weight_kg_adm"], s=14, c="red", label="died")
a.set_xlabel("Age (months)"); a.set_ylabel("Weight (kg)"); a.set_title("Weight depends on age"); a.legend()

a = ax.flat[5]
m = (died.groupby(df["bcsmotor_adm"]).mean() * 100).sort_values()
a.barh(m.index.str[:22], m.values); a.set_xlabel("Mortality %"); a.set_title("Mortality by Blantyre motor")
plt.tight_layout(); plt.show()

## ✍️ Try it yourself
1. Change the column in section 8 to `feedingstatus_adm` — does "not feeding at all" predict death?
2. In section 10, replace lactate with `spo2site1_pc_oxi_adm`. What do you see?
3. What is mortality among children with `symptoms_adm___15` (coma) = "Checked"?

**Next — P2:** feature engineering (WHO z-scores, encoding, Blantyre total, removing leakage, cleaning impossible values).